In [51]:
# import required libs
import numpy as np
import sys
sys.path.append(r"../Communiaction")
sys.path.append(r"../Utils")
import TCP_IP_UTILS
import Utils
import time

In [52]:
#Add power supply code here -> DC Sources
import sys
import pyvisa
sys.path.append(r"../PowerSupply")
import PS_Utils
Device_ID_DC = "USB0::0x2A8D::0x1202::MY59003914::0::INSTR"
rm = pyvisa.ResourceManager()
resources = rm.list_resources()
print("Available VISA resources:")
for r in resources:
        print(" -", r)
device_dc = PS_Utils.connect_E36313A(Device_ID_DC)
DIG_LS = 1
ANA_LS = 2
MAIN = 3

Available VISA resources:
 - USB0::0x0957::0x1F01::MY53270560::0::INSTR
 - USB0::0x0957::0x1F01::MY57280340::0::INSTR
 - USB0::0x2A8D::0x1202::MY59003914::0::INSTR
 - USB0::0x2A8D::0x9007::MY62310119::0::INSTR
 - USB0::0x2A8D::0x9201::MY61391394::0::INSTR
Connected to: Keysight Technologies,E36313A,MY59003914,2.1.0-1.0.4-1.12


In [53]:
#connect to ethernet socket 
HOST = '192.168.1.10'  # Remote device IP (server IP)
PORT = 7               # Echo port
client_socket = TCP_IP_UTILS.tcp_connect(HOST,PORT)
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)
time.sleep(1) #Wait for IOs to settle

[Client] Connected to 192.168.1.10:7
Default Signals Set
[128]


In [54]:
#First Turn on Digital Leval Shifters
PS_Utils.set_voltage_E36313A(device_dc,DIG_LS,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,DIG_LS))
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

['+2.99835900E+00', '+5.48100000E-03']
Default Signals Set
[128]


In [55]:
#Turn on Main Chip PS
PS_Utils.set_voltage_E36313A(device_dc,MAIN,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,MAIN))

['+2.99818600E+00', '+1.26680000E-02']


In [56]:
#Turn on CLK 
#Device_ID_CLK = "USB0::0x0957::0x1F01::MY57280340::0::INSTR"
Device_ID_CLK = "USB0::0x0957::0x1F01::MY53270560::0::INSTR"
device_clk = PS_Utils.connect_N5173B(Device_ID_CLK)
PS_Utils.set_voltage_N5173B(device_clk,0.45)
PS_Utils.set_frequency_N5173B(device_clk,1000000000)
PS_Utils.enable_output_N5173B(device_clk)

Connected to: Agilent Technologies, N5173B, MY53270560, B.01.70


In [57]:
def generate_binary_array(length, percent_ones):
    num_ones = int(length * percent_ones / 100)
    num_zeros = length - num_ones
    
    # Create array with required number of 1s and 0s
    arr = np.array([1] * num_ones + [0] * num_zeros, dtype=np.uint8)
    
    # Shuffle randomly
    np.random.shuffle(arr)
    
    return arr.tolist()

In [58]:
#Scan In the Data in WRITE_SC 
scan_len = Utils.WRITE_DATA_SC["len"]
scan_id = Utils.WRITE_DATA_SC["id"]
scan_data = generate_binary_array(scan_len,50)
print(scan_data)
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
print(ret_data)

[1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1]
[40, 1]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 3
Received Data Size: 1
All Data Recei

In [59]:
#Turn on WRITE_EN and READ_EN. SA Now sees WRITE values as inputs CS = 0
signal_array = [Utils.WRITE_EN,Utils.READ_EN]
value_array = [1,1]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
print(ret_data)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
[128]


In [60]:
def set_CS(cs):
    ret_data = 0
    if cs == 0:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [0,0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 1:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [1,0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 2:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [0,1]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 3:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [1,1]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    return ret_data

In [61]:

for cs in range(0,4):
    #select the BL
    ret_data = set_CS(cs)
    #Turn On SA_EN
    signal_array = [Utils.SA_EN]
    value_array = [0]
    ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    #Sample the SA output
    signal_array = [Utils.CLK_SA]
    value_array = [0]
    ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    #Turn of SA and CLK_SA
    signal_array = [Utils.SA_EN,Utils.CLK_SA]
    value_array = [1,1]
    ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Fn identifier Sent
Received: 128
All 

In [62]:
#set signals to default
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

Default Signals Set
[128]


In [63]:
#Set scan chain to SA_OUT. Load inputs to scan chain
signal_array = [Utils.SCN_SEL_2,Utils.SCN_SEL_1,Utils.SCN_SEL_0]
value_array = Utils.SA_OUT_SC["value"] #Scan chain id
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
print(ret_data)
#IN_EN = High CLK_A = High SCN_IN = LOW
signal_array = [Utils.IN_EN,Utils.CLK_A,Utils.SCN_IN]
value_array = [1,1,1]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
#CLK_A = LOW 
signal_array = [Utils.CLK_A]
value_array = [0]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
#IN_EN = LOW 
signal_array = [Utils.IN_EN]
value_array = [0]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)

Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
[128]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 1
Received Data Size: 1
All Data Received


In [64]:
#SCAN_OUT SA_OUT_SC
scan_id = Utils.SA_OUT_SC["id"]
scan_len = Utils.SA_OUT_SC["len"]
sc_out = Utils.ScanOUT_Data(client_socket,scan_id,scan_len,1)
print(sc_out)

[40, 1]
Fn identifier Sent
Received: 128
All Data Sent
Received: 128
FN: 4
Received Data Size: 40
All Data Received
[215, 176, 52, 84, 164, 123, 91, 22, 172, 242, 149, 237, 228, 31, 209, 36, 182, 212, 107, 64, 182, 132, 139, 182, 255, 89, 166, 24, 66, 100, 194, 238, 21, 146, 87, 104, 188, 85, 85, 85]


In [65]:
#set signals to default
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

Default Signals Set
[128]


In [66]:
u8_array = []
for i in range(0, len(scan_data), 8):
    byte_bits = scan_data[i:i+8]
    value = sum(bit << j for j, bit in enumerate(byte_bits))  # index 0 is LSB
    u8_array.append(value)
print(sc_out)
print(u8_array)
print(u8_array == sc_out[:-3])

[215, 176, 52, 84, 164, 123, 91, 22, 172, 242, 149, 237, 228, 31, 209, 36, 182, 212, 107, 64, 182, 132, 139, 182, 255, 89, 166, 24, 66, 100, 194, 238, 21, 146, 87, 104, 188, 85, 85, 85]
[215, 176, 52, 84, 164, 123, 91, 22, 172, 242, 149, 237, 228, 31, 209, 36, 182, 212, 107, 64, 182, 132, 139, 182, 255, 89, 166, 24, 66, 100, 194, 238, 21, 146, 87, 104, 188]
True


In [67]:
PS_Utils.kill_channel_E36313A(device_dc,ANA_LS)
time.sleep(0.3)
PS_Utils.kill_channel_E36313A(device_dc,MAIN)
time.sleep(0.3)
PS_Utils.kill_channel_E36313A(device_dc,DIG_LS)
PS_Utils.disconnect_E36313A(device_dc)
time.sleep(0.3)
PS_Utils.kill_N5173B(device_clk)
PS_Utils.disconnect_N5173B(device_clk)

0